In [ ]:
%load_ext autoreload
%autoreload 2
    
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdate
import datetime as dt
from tqdm import tqdm
from glob import glob
import re
from pathlib import Path
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

import dunestyle.matplotlib as dunestyle
from cycler import cycler
plt.rcParams.update( { "axes.prop_cycle":cycler(color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']) })
        
import sys
sys.path.insert(1, '../')
from waffles.np02_utils.AutoMap import getModuleName
from utils import PATH_XE_OUTPUTS
from utils_monitoring import load_database, load_fit_results, make_relative_cols, load_injections, load_results, add_ppm_per_run, database_injections_per_run, check_health
from utils_monitoring import plot_vs_time_per_channel, iplot, execute_by_module

In [ ]:
dfg = load_fit_results(f"{PATH_XE_OUTPUTS}/global_analysis_complete-hvieirad/", sufix='global_')
xedb = load_database()
check_health(dfg, xedb, split_modules=False)
dfg

In [ ]:
df_injections = load_injections()
df_injections

In [ ]:
df = add_ppm_per_run(dfg)
df

In [ ]:
df_first = df.groupby('run').first().reset_index()
df_first['module'] = 'X'

In [ ]:
df_first['1/tta'] = 1/df_first['tta']
df_first['1/ttx'] = 1/df_first['ttx']
df_first['err1/tta'] = df_first['errtta']/df_first['tta']**2
df_first['err1/ttx'] = df_first['errttx']/df_first['ttx']**2

# df_first = df_first.loc[~((df_first['1/tta'] > 20) & (df_first['ppm'] > 1))]

In [ ]:
modules_ok = ['X']
x = 'time'
y = 'ppm'

fig = go.Figure()
execute_by_module(df_first, iplot, fig=fig, y=y, x=x, modules=modules_ok, df_injections=df_injections, name=r'runs', withError=False)
fig.update_layout(template="plotly_white",
                  yaxis_title=r'Concentration [ppm]'
                 )
fig.show()

In [ ]:
modules_ok = ['X']
x = 'time'

fig = go.Figure()
execute_by_module(df_first, iplot, fig=fig, y='tta', x=x, modules=modules_ok, df_injections=df_injections, name=r'$\tau_{TA}$', withError=True)
execute_by_module(df_first, iplot, fig=fig, y='ttx', x=x, modules=modules_ok, df_injections=df_injections, name=r'$\tau_{TX}$', withError=True)
fig.update_layout(template="plotly_white",
                  yaxis_title=r'$\tau\, [\mu\text{s}]$',
                 )

fig.show()
# fig.write_html("tau_vs_time.html")


In [ ]:
modules_ok = ['X']
x = 'ppm'

fig = go.Figure()
execute_by_module(df_first, iplot, fig=fig, y='1/tta', x=x, modules=modules_ok, df_injections=df_injections, name=r'$\tau_{TA}$', withError=True, fontsize=18)
execute_by_module(df_first, iplot, fig=fig, y='1/ttx', x=x, modules=modules_ok, df_injections=df_injections, name=r'$\tau_{TX}$', withError=True, fontsize=18)
fig.update_layout(template="plotly_white",
                  yaxis_title=r'$1/ \tau\, [\mu\text{s}^{-1}]$',
                  font=dict(size=18),
                  width=1000,
                  height=500
                 )

# fig.write_html("1_over_tau_vs_time.html")

fig.show()

In [ ]:
def ttxvalid(df):
    return df[df['ttx']>0]
plt.figure(figsize=(8,4))
# plot_vs_time_per_channel(df, y='t3[ns]',  module="C3(1)", xlim=(pd.to_datetime('2025-09-19 8:00'), pd.to_datetime('2025-09-19 16:30')), showhours=True, label="C3(1)")
xlim = (pd.to_datetime('2026-03-22 0:00'), pd.to_datetime('2026-04-25 16:30'))
xlim = (pd.to_datetime('2026-03-23 13:00:00'), pd.to_datetime('2026-03-24 16:00'))
plot_vs_time_per_channel(df_first, y='ttx',  module="X", xlim=xlim, showhours=True, selection=ttxvalid, label=r"$\tau_{TX}$", df_injections=None, autocolor=False, kwargs_errorbar=dict(color='C0', marker='.'))
plot_vs_time_per_channel(df_first, y='tta',  module="X", xlim=xlim, showhours=True, label=r"$\tau_{TA}$", df_injections=df_injections, autocolor=False, kwargs_errorbar=dict(color='C1', marker='.'))
plt.ylabel(r'$\tau$ [$\mu$s]')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
dft = df_first[df_first['ppm'] < 0.025]
plt.errorbar(dft['ppm'], dft['ttx'], xerr=dft['ppm']*0.01, yerr=dft['errttx'], ls='', markersize=10, marker='.', label=r'$\tau_{TX}$')
plt.errorbar(dft['ppm'], dft['tta'], xerr=dft['ppm']*0.01, yerr=dft['errtta'], ls='', markersize=10, marker='.', label=r'$\tau_{TA}$')
plt.ylabel(r'1/$\tau$ [$\mu$s$^{-1}$]')
plt.xlabel(r'Concentration [ppm]')
plt.legend()
# plt.xlim(0.8,None)

In [ ]:
plt.figure(figsize=(14,6))
# plot_vs_time_per_channel(df, y='t3[ns]',  module="C3(1)", xlim=(pd.to_datetime('2025-09-19 8:00'), pd.to_datetime('2025-09-19 16:30')), showhours=True, label="C3(1)")
# xlim = (pd.to_datetime('2026-03-22 0:00'), pd.to_datetime('2026-04-27 12:30'))
plot_vs_time_per_channel(df_first, y='ppm',  module="X", xlim=xlim, showhours=True, label=r"", df_injections=None, autocolor=False, kwargs_errorbar=dict(color='C0', marker='.'))
plt.show()

In [ ]:
dt